In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l7.4 QLoRA

QLoRA stacks two tricks: keep the frozen base in **4-bit** to save memory, and
train a LoRA adapter on top in full precision. It is what lets people fine-tune
large models on a single GPU.

> CI note: real QLoRA needs a 4-bit GPU kernel (bitsandbytes). Here we *simulate*
> the precision loss on CPU with `fake_quantize`, so the lesson runs anywhere.

In [ ]:
from pocketlm import (train_tiny, pick_config, fake_quantize, add_lora,
                      trainable_parameters, make_batch)

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
with torch.no_grad():
    for p in model.parameters():
        p.copy_(fake_quantize(p, bits=4))   # simulate the 4-bit frozen base
add_lora(model, rank=4)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)
opt = torch.optim.AdamW(trainable_parameters(model), lr=1e-3)
g = torch.Generator().manual_seed(0)
for _ in range(60 if os.environ.get("POCKETLM_CI") else 150):
    x, y = make_batch(data, model.cfg.block_size, 16, generator=g)
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
print("adapter trains over a 4-bit base, final loss:", round(loss.item(), 3))

The quantized base loses a little accuracy, but the adapter recovers task
performance, and the memory footprint of the base roughly quarters. That trade is
why QLoRA democratized fine-tuning.

In [ ]:
assert torch.isfinite(loss)